# QuickBite — Fast-Food Market Basket Analysis & Recommendation System
## Academic Machine Learning Practical Project (A.Y. 2025–26)
---
### 1. Introduction and Objectives
Market Basket Analysis (MBA) is an unsupervised learning and association rule mining technique designed to uncover affinities between items co-purchased in transactional databases. In fast-food retail, identifying item combinations enables data-backed combo meal design, targeted kiosk upselling, and optimized inventory management.

### 2 & 3. Dataset Source, Provenance, and Loading
We evaluate two datasets:
1. **QuickBite Benchmark Fast-Food Dataset** (Synthetic Demonstration Data, 5,000 orders).
2. **Chipotle Mexican Grill Orders** (Real-World Restaurant Transaction Data, 1,834 orders).

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import TransactionPreprocessor
from src.apriori_manual import ManualApriori, explain_manual_worked_example
from src.association_analysis import AssociationMiner, benchmark_apriori_vs_fpgrowth, run_parameter_tuning_grid
from src.recommendations import ComboRecommendationEngine

# Load synthetic fast-food benchmark data
data_path = os.path.join('..', 'data', 'quickbite_synthetic_orders.csv')
df_raw = pd.read_csv(data_path)
print(f'Loaded dataset with {len(df_raw)} records across {df_raw["order_id"].nunique()} orders.')
df_raw.head()

### 4 & 5. Dataset Dimensions, Schema, Missing-Value, and Duplicate Diagnostics

In [ ]:
preprocessor = TransactionPreprocessor(order_col='order_id', item_col='item_name')
diagnostics = preprocessor.inspect_dataset(df_raw)
print('Dataset Quality Audit:')
for k, v in diagnostics.items():
    if k not in ['missing_values', 'dtypes', 'columns']:
        print(f' - {k}: {v}')

print('\nMissing values per column:')
print(pd.Series(diagnostics['missing_values']))

### 6 & 7. Exploratory Data Analysis & Data Preprocessing
Data cleaning includes whitespace stripping, case normalization, and removal of invalid records.

In [ ]:
df_clean = preprocessor.clean_dataset(df_raw)
print('Cleaning summary:', preprocessor.cleaning_report)

# Top 10 items frequency
top_items = df_clean['item_name'].value_counts().head(10)
plt.figure(figsize=(10, 4))
sns.barplot(x=top_items.values, y=top_items.index, palette='Oranges_r')
plt.title('Top 10 Fast-Food Menu Items by Order Count')
plt.xlabel('Frequency')
plt.show()

### 8. Transaction Basket Construction & One-Hot Encoding
We construct both item transaction lists and binary one-hot encoded presence matrices.

In [ ]:
trans_list, one_hot_df = preprocessor.create_transaction_baskets(df_clean)
stats = preprocessor.get_basket_statistics(df_clean, trans_list)
print(f'Constructed {len(trans_list)} baskets. Average basket size: {stats["avg_basket_size"]} items.')
print('One-hot shape:', one_hot_df.shape)
one_hot_df.head(3)

### 9. First-Principles Manual Apriori Implementation (Mandatory viva section)
Runs `src/apriori_manual.py`, built entirely from scratch without external mining libraries.

In [ ]:
manual_miner = ManualApriori(min_support=0.03, min_confidence=0.20, min_lift=1.0)
manual_miner.fit(trans_list)
man_itemsets = manual_miner.get_frequent_itemsets_df()
man_rules = manual_miner.get_association_rules_df()
print(f'Manual Apriori discovered {len(man_itemsets)} itemsets and {len(man_rules)} rules.')
man_rules.head(5)

### 10 & 11. Library-Based Apriori and FP-Growth Mining (mlxtend)

In [ ]:
miner = AssociationMiner(min_support=0.03, min_confidence=0.20, min_lift=1.0)
ap_itemsets, ap_time = miner.mine_frequent_itemsets(one_hot_df, algorithm='apriori')
fp_itemsets, fp_time = miner.mine_frequent_itemsets(one_hot_df, algorithm='fpgrowth')
print(f'Apriori execution time: {ap_time:.4f}s | FP-Growth execution time: {fp_time:.4f}s')

### 12. Association Rule Generation & Metric Extraction

In [ ]:
rules_df = miner.generate_rules(fp_itemsets, num_transactions=len(one_hot_df))
print(f'Total qualified rules: {len(rules_df)}')
rules_df[['antecedent_str', 'consequent_str', 'support', 'confidence', 'lift', 'leverage']].head(8)

### 13. Parameter Tuning Experiments
Systematically tests support thresholds [0.01, 0.02, 0.05, 0.08] across confidence levels [0.2, 0.4].

In [ ]:
tuning_results = run_parameter_tuning_grid(
    one_hot_df,
    support_grid=[0.01, 0.02, 0.03, 0.05, 0.08],
    confidence_grid=[0.20, 0.35, 0.50],
    algorithm='fpgrowth'
)
tuning_results.head(10)

### 14. Manual Metric Calculations & Worked Numerical Example (Viva Voce)
We verify metric formulas using a known reference: Burger (40%), Fries (30%), Both (20%).

In [ ]:
worked_ex = explain_manual_worked_example()
for k, v in worked_ex.items():
    if k != 'interpretation':
        print(f'{k}: {v}')
print('\nInterpretation:', worked_ex['interpretation'])

### 15. Empirical & Theoretical Algorithm Comparison

In [ ]:
bench = benchmark_apriori_vs_fpgrowth(one_hot_df, min_support=0.03, min_confidence=0.20)
print(bench['insights'])
bench['comparison_df']

### 16. Visual Analytics: Support vs Confidence & Lift Distribution

In [ ]:
if not rules_df.empty:
    plt.figure(figsize=(10, 5))
    scatter = plt.scatter(
        rules_df['support'],
        rules_df['confidence'],
        c=rules_df['lift'],
        cmap='Oranges',
        s=rules_df['lift'] * 25,
        edgecolor='black',
        alpha=0.85
    )
    plt.colorbar(scatter, label='Lift')
    plt.title('Support vs. Confidence (Size & Color = Lift)')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

### 17 & 18. Business Interpretation, Limitations, and Conclusion
**Key Findings:**
- Strongest affinity pairs naturally align around Burger + Large French Fries + Soda.
- Nuggets heavily trigger dipping sauce cross-sells (BBQ Dip).
- **Recommendation:** Bundle high-lift combos into discounted meal deals; configure POS cash registers and digital kiosks with dynamic prompts.

**Limitations:**
- Association rules demonstrate co-occurrence correlation, not causal purchasing intent.
- Equal weight is assigned to all items regardless of margin contribution; future work should explore High Utility Itemset Mining (HUIM).